### 4.4 Definição do Hardware

In [ ]:
# Verificar o status da GPU para garantir que o Colab alocou a placa de vídeo
!nvidia-smi

### 4.5 Configuração do Ambiente (Faster R-CNN e Dependências)

In [1]:
import torch
import torchvision
print(f"PyTorch Versão: {torch.__version__} | Torchvision Versão: {torchvision.__version__}")

PyTorch Versão: 2.11.0+cu128 | Torchvision Versão: 0.26.0+cu128


### Conexão com a Nuvem (Google Drive)

In [3]:
from google.colab import drive
import os

# Montar o Google Drive para ler o dataset e salvar os resultados
drive.mount('/content/drive')

KeyboardInterrupt: 

### 4.3 Pré-processamento dos Datasets

In [5]:
import os

def listar_estrutura(pasta_raiz, nivel_max=3):
    print(f"📁 Estrutura de pastas em: {pasta_raiz}\n")
    for raiz, diretorios, arquivos in os.walk(pasta_raiz):
        nivel = raiz.replace(pasta_raiz, '').count(os.sep)
        if nivel >= nivel_max:
            continue
        indentacao = ' ' * 4 * (nivel)
        print(f"{indentacao}├── {os.path.basename(raiz)}/")
        sub_arquivos = [f for f in arquivos if not f.startswith('.')]
        if sub_arquivos:
            print(f"{indentacao}│   └── (Contém {len(sub_arquivos)} arquivos. Exemplo: {sub_arquivos[0]})")

# Executa para o FasterRCNN
if os.path.exists("/content/FasterRCNN_dataset"):
    listar_estrutura("/content/FasterRCNN_dataset")

📁 Estrutura de pastas em: /content/FasterRCNN_dataset

├── FasterRCNN_dataset/
    ├── annotations/
    │   └── (Contém 3 arquivos. Exemplo: train.json)
    ├── images/
        ├── test/
        │   └── (Contém 1034 arquivos. Exemplo: 000000501801.jpg)
        ├── val/
        │   └── (Contém 1033 arquivos. Exemplo: 000000325720.jpg)
        ├── train/
        │   └── (Contém 8265 arquivos. Exemplo: 000000094097.jpg)


In [ ]:
import os
import shutil
import torchvision
from torchvision.transforms import transforms as T
import torch

# ==============================================================================
# 1. CAMINHOS E DESCOMPACTAÇÃO DIRETA NA VM
# ==============================================================================
caminho_zip_drive = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/FasterRCNN_dataset.zip"
pasta_local_vm = "/content/FasterRCNN_dataset"
zip_local_vm = "/content/FasterRCNN_dataset.zip"

print("🔍 Verificando arquivo no Drive...")
if not os.path.exists(caminho_zip_drive):
    raise FileNotFoundError(f"❌ Arquivo não encontrado no Drive: {caminho_zip_drive}")

# Limpar tentativas e resíduos anteriores na VM local
if os.path.exists(zip_local_vm):
    os.remove(zip_local_vm)
if os.path.exists(pasta_local_vm):
    shutil.rmtree(pasta_local_vm)

os.makedirs(pasta_local_vm, exist_ok=True)

print("🚚 Copiando o FasterRCNN_dataset.zip do Drive para o SSD local da VM...")
shutil.copy2(caminho_zip_drive, zip_local_vm)
print(f"📦 Arquivo copiado com sucesso ({os.path.getsize(zip_local_vm) / (1024**3):.2f} GB).")

print("⚡ Descompactando diretamente na VM usando o utilitário nativo...")
# Usamos o unzip nativo do Linux do Colab que é muito mais performático para ficheiros grandes
!unzip -q {zip_local_vm} -d {pasta_local_vm}

# Remove o zip local para libertar espaço em disco na VM, mantendo apenas a pasta extraída
if os.path.exists(zip_local_vm):
    os.remove(zip_local_vm)
print("✅ Descompactação concluída na VM!")


# ==============================================================================
# 2. DEFINIÇÃO DO DATASET (COCO FORMAT) E DATALOADERS
# ==============================================================================

# Classe Customizada utilizando o CocoDetection nativo do PyTorch
class FasterRCNNCocoDataset(torchvision.datasets.CocoDetection):
    def __init__(self, root, annFile, transforms=None):
        super(FasterRCNNCocoDataset, self).__init__(root, annFile)
        self.transforms = transforms

    def __getitem__(self, idx):
        # O CocoDetection nativo retorna (imagem_pil, lista_de_anotacoes_json)
        img, target_orig = super(FasterRCNNCocoDataset, self).__getitem__(idx)
        
        boxes = []
        labels = []
        
        for ann in target_orig:
            # O COCO guarda as caixas como [xmin, ymin, largura, altura]
            bbox = ann['bbox']
            xmin = bbox[0]
            ymin = bbox[1]
            xmax = xmin + bbox[2]  # Converte largura para xmax
            ymax = ymin + bbox[3]  # Converte altura para ymax
            
            # Garantir integridade da box (área maior que zero para não quebrar a RPN)
            if xmax > xmin and ymax > ymin:
                boxes.append([xmin, ymin, xmax, ymax])
                # Vincula o ID da classe diretamente (o Faster R-CNN reserva o 0 para background)
                labels.append(ann['category_id'])
        
        # Se a imagem não contiver objetos válidos, criamos tensores vazios estruturados
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        target["image_id"] = torch.tensor([idx])
        
        if self.transforms:
            img = self.transforms(img)
            
        return img, target

# Transformação padrão para as imagens (conversão para Tensor do PyTorch)
dataset_transforms = T.Compose([
    T.ToTensor(),
])

# Instanciação mapeando corretamente as subpastas extraídas na VM
train_dataset = FasterRCNNCocoDataset(
    root=os.path.join(pasta_local_vm, "images/train"),
    annFile=os.path.join(pasta_local_vm, "annotations/train.json"),
    transforms=dataset_transforms
)

val_dataset = FasterRCNNCocoDataset(
    root=os.path.join(pasta_local_vm, "images/val"),
    annFile=os.path.join(pasta_local_vm, "annotations/val.json"),
    transforms=dataset_transforms
)

test_dataset = FasterRCNNCocoDataset(
    root=os.path.join(pasta_local_vm, "images/test"),
    annFile=os.path.join(pasta_local_vm, "annotations/test.json"),
    transforms=dataset_transforms
)

# Função de agrupamento necessária para lidar com tamanhos de imagem dinâmicos no Faster R-CNN
def collate_fn(batch):
    return tuple(zip(*batch))

# Criação final dos DataLoaders (Batch size = 8 para evitar estouro de memória da GPU)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)

print("=" * 50)
print(f"📊 Amostras de Treino: {len(train_dataset)} imagens")
print(f"📊 Amostras de Validação: {len(val_dataset)} imagens")
print(f"📊 Amostras de Teste: {len(test_dataset)} imagens")
print("=" * 50)

loading annotations into memory...
Done (t=0.92s)
creating index...
index created!
loading annotations into memory...
Done (t=0.10s)
creating index...
index created!
loading annotations into memory...
Done (t=0.11s)
creating index...
index created!
✅ DataLoaders criados com sucesso!
📊 Amostras de Treino: 8265 imagens
📊 Amostras de Validação: 1033 imagens
📊 Amostras de Teste: 1034 imagens


### 4.6 Definição de Hiperparâmetros e 4.7 Treinamento do Modelo

In [ ]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from datetime import datetime
from tqdm import tqdm

# 1. Criação do diretório de saída estruturado no seu Drive
agora = datetime.now()
nome_treino_rcnn = agora.strftime("Treino_faster_rcnn_%Y-%m-%d_%Hh%M")
diretorio_drive_rcnn = "/content/drive/MyDrive/Classroom/TCC_Kevin_Rafael/Colab/Resultados_FasterRCNN"
os.makedirs(f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights", exist_ok=True)

print(f"📁 Destino dos resultados: {diretorio_drive_rcnn}/{nome_treino_rcnn}")

# 2. Inicializar o Faster R-CNN com Pesos Pré-treinados (Transfer Learning)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)

# Substituir o cabeçalho de classificação para adequar ao número de classes do seu TCC
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
model.to(device)

# 3. Otimizador padrão de Two-Stage Detectors
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
epochs = 1 # Mantendo o mesmo padrão de épocas dos testes anteriores
best_loss = float('inf')

# 4. Loop de Treinamento Principal
for epoch in range(epochs):
    model.train()
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for images, targets in progress_bar:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Ignorar amostras sem anotações para não quebrar a Region Proposal Network (RPN)
        if any(len(t["boxes"]) == 0 for t in targets):
            continue

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
        progress_bar.set_postfix(loss=losses.item())
        
    media_loss = epoch_loss / len(train_loader)
    print(f"📉 Fim da Epoch {epoch+1} - Loss Combinada: {media_loss:.4f}")
    
    # Salvamento Periódico Automatizado (last.pt)
    torch.save(model.state_dict(), f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights/last.pt")
    
    # Salvamento Baseado em Performance (best.pt)
    if media_loss < best_loss:
        best_loss = media_loss
        torch.save(model.state_dict(), f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights/best.pt")
        print("🏆 Excelente! Melhor modelo atualizado no Drive.")

print("✅ Treinamento do Faster R-CNN concluído!")

### 4.7 Inferência Prática e Avaliação de Métricas (Conjunto de Testes)

In [ ]:
def calcular_precisao_rcnn(model, data_loader, device):
    model.eval()
    total_corretas = 0
    total_propostas = 0
    
    with torch.no_grad():
        for images, targets in data_loader:
            images = list(img.to(device) for img in images)
            outputs = model(images)
            
            for i, output in enumerate(outputs):
                labels_reais = targets[i]["labels"].tolist()
                pred_labels = output["labels"].tolist()
                scores = output["scores"].tolist()
                
                # Considerar apenas predições com alto grau de certeza (Score > 0.5)
                confiáveis = [pred_labels[j] for j, s in enumerate(scores) if s > 0.5]
                
                for lbl in confiáveis:
                    if lbl in labels_reais:
                        total_corretas += 1
                total_propostas += len(confiáveis)
                
    return total_corretas / total_propostas if total_propostas > 0 else 0

# Carregar os pesos calibrados do melhor treino para computar os resultados finais
model.load_state_dict(torch.load(f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/weights/best.pt"))
precisao_final = calcular_precisao_rcnn(model, test_loader, device)

print("=" * 60)
print(f"📊 Avaliação Final no Conjunto de Testes - Faster R-CNN")
print(f"🎯 Precisão Média Obtida: {precisao_final * 100:.2f}%")
print("=" * 60)

# Persistir o sumário de performance em TXT na pasta de resultados do Drive
with open(f"{diretorio_drive_rcnn}/{nome_treino_rcnn}/resultado_teste.txt", "w") as f:
    f.write("Métricas de Avaliação Final - Faster R-CNN (ResNet50 v2)\n")
    f.write(f"Precisão Global: {precisao_final * 100:.2f}%\n")
print("💾 Arquivo contendo as métricas gerado com sucesso no Drive!")